# 01 · Exploración rápida: ¿qué estrategias funcionan mejor?

Analizamos la **temporada 0 (clasificatoria)**, en la que cada estrategia juega contra todas las demás. Es la comparación más justa: todas tienen los mismos rivales.

Tablas disponibles en `result`:
- `matches`: un partido por fila (`season, league, A, B, pa, pb, seed`)
- `standings`: clasificación por temporada y liga
- `strategy_stats`: cuántas veces jugó SPLIT/STEAL cada estrategia en cada liga

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from split_steal.runner import load_run
from split_steal import plots

# Run canónico versionado en el repo. Para regenerarlo:
#   split-steal tournament --name canonical --out-dir results/canonical
RUN_DIR = Path("../results/canonical/canonical")
result = load_run(RUN_DIR)
result.config

TournamentConfig(roster='full', n_leagues=5, n_per_league=6, rounds=100, n_seasons=20, n_ascend_descend=2, seed=42, qualifying=True)

## Resumen por estrategia

`summarize` desdobla cada partido en dos filas (una por jugador) y agrega por estrategia. Ojo: **ganar partidos no es lo mismo que puntuar**; `AlwaysSteal` nunca pierde un partido pero puntúa poco.

In [2]:
from split_steal.cli import summarize

qualifying = result.matches[result.matches["season"] == 0]
summary = summarize(qualifying)
summary.round(1)

,name,matches,wins,losses,ties,mean_payoff,median_payoff
0,GrimTrigger,29,9,4,16,275.5,300.0
1,SplitUntilBetrayedTwice,29,7,5,17,271.3,300.0
2,ForgiveAfter3,29,8,4,17,264.6,300.0
3,TwoTitsForTat,29,9,4,16,258.8,300.0
4,ForgivingTitForTat,29,0,12,17,258.5,300.0
5,CopycatDelayed,29,0,12,17,253.7,300.0
6,ConditionalRandom,29,0,12,17,253.6,300.0
7,ProbTitForTat,29,0,12,17,252.1,300.0
8,RevengeOneRound,29,0,9,20,252.1,300.0
9,TitForTat,29,0,8,21,252.1,300.0


## Puntos por ronda en la clasificatoria

Las líneas marcan lo que se gana por ronda con cooperación mutua (3) y traición mutua (1).

In [3]:
plots.plot_qualifying_points(result);

## ¿Cooperar más hace ganar más?

Las mejores estrategias cooperan mucho pero **no siempre**: castigan la traición. `AlwaysSplit` coopera el 100 % y queda a media tabla porque la explotan.

In [4]:
plots.plot_cooperation_vs_payoff(result);

## Distribución de puntos por partido

Cada caja resume los 29 partidos de una estrategia en la clasificatoria.

In [5]:
side_a = qualifying[["A", "pa"]].rename(columns={"A": "name", "pa": "score"})
side_b = qualifying[["B", "pb"]].rename(columns={"B": "name", "pb": "score"})
scores = pd.concat([side_a, side_b], ignore_index=True)
order = scores.groupby("name")["score"].median().sort_values().index

fig, ax = plt.subplots(figsize=(8, 9))
ax.boxplot([scores.loc[scores.name == n, "score"] for n in order], vert=False,
           tick_labels=list(order), widths=0.6)
ax.set_xlabel("Puntos por partido (100 rondas)")
ax.set_title("Puntos por partido en la clasificatoria");